In [1]:
%load_ext autoreload
%autoreload 2
import pyepo
import numpy as np
import random
import torch

In [2]:
# fix random seed
random.seed(135)
np.random.seed(135)
torch.manual_seed(135)

In [3]:
def visLearningCurve(loss_log, regret_log):
    # create figure and subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16,4))

    # draw plot for training loss
    ax1.plot(loss_log, color="c", lw=2)
    ax1.tick_params(axis="both", which="major", labelsize=12)
    ax1.set_xlabel("Iters", fontsize=16)
    ax1.set_ylabel("Loss", fontsize=16)

    # draw plot for regret on test
    ax2.plot(regret_log, color="royalblue", ls="--", alpha=0.7, lw=2)
    ax2.set_xticks(range(0, len(regret_log), 2))
    ax2.tick_params(axis="both", which="major", labelsize=12)
    ax2.set_ylim(0, 0.5)
    ax2.set_xlabel("Epochs", fontsize=16)
    ax2.set_ylabel("Regret", fontsize=16)

    plt.title("Learning Curve on Training Set", fontsize=16)
    plt.show()

In [4]:
# generate data for 2D knapsack
K = 100
capacities = [K]
m = 275 # number of items
n = 100 # number of data
p = 5 # size of feature
big_feature = m*p
deg = 6 # polynomial degree
dim = 1 # dimension of knapsack
noise_width = 0.5 # noise half-width
weights, x, c = pyepo.data.knapsack.genData(n, m*p, m, deg=deg, dim=dim, noise_width=noise_width)

In [5]:
# data split
from sklearn.model_selection import train_test_split
x_train, x_test, c_train, c_test = train_test_split(x, c, test_size=90, random_state=246)

In [6]:
optmodel = pyepo.model.grb.knapsackModel(weights, capacities)

Restricted license - for non-production use only - expires 2026-11-23


In [7]:
# get training data set
dataset_train = pyepo.data.dataset.optDataset(optmodel, x_train, c_train,)
# get test data set
dataset_test = pyepo.data.dataset.optDataset(optmodel, x_test, c_test)

Optimizing for optDataset...


100%|██████████| 10/10 [00:00<00:00, 269.27it/s]


Optimizing for optDataset...


100%|██████████| 90/90 [00:03<00:00, 24.80it/s]


In [8]:
# get data loader
from torch.utils.data import DataLoader
batch_size = 1
loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

In [1]:
import torch
import torch.nn as nn

######################################################
# 1) Custom Autograd Function for ratio_rating
######################################################

class RatioRatingFunction(torch.autograd.Function):
    """
    A custom Function that:
      - Takes your model parameters (rolled up) and any other inputs
      - Samples from the distribution
      - Computes ratio_rating
      - On backward, uses forward-mode jacobian * ratio_rating for the gradient wrt params
    """

    @staticmethod
    def forward(ctx,
                params_single_tensor,      # (param_dim,) single rolled-up param vector
                feat,                      # input features for this time step or batch
                time,
                M_score_func,
                model_build_from_params):
        """
        forward: returns ratio_rating_TS as a normal tensor that autograd can track.
        """

        with torch.no_grad():
            # 1) Rebuild the distribution from the param vector
            dist = model_build_from_params(params_single_tensor, feat, time)

            # 2) Sample from the distribution
            #    Suppose we get shape [M_score_func, B, D], for example
            y_sample_MBD = dist.sample((M_score_func,))
            # reorder dims so it's [B, M_score_func, D] if you like
            y_sample_BMD = y_sample_MBD.permute(1,0,2)  

            # 3) ratio_rating_BMD = y_sample / (sum(...) + 1)
            denom_BM1 = y_sample_BMD.sum(dim=-1, keepdim=True) + 1.0
            ratio_rating_BMD = y_sample_BMD / denom_BM1

            # We'll just average over M to get ratio_rating_BD
            ratio_rating_BD = ratio_rating_BMD.mean(dim=1)

        # 4) We must save everything needed for backward
        #    - the original param vector
        #    - the sample we used
        #    - the ratio_rating_BMD to help us replicate your jacobian logic
        #    - the function needed to rebuild distributions
        ctx.save_for_backward(params_single_tensor, y_sample_BMD, ratio_rating_BMD)
        ctx.model_build_from_params = model_build_from_params
        ctx.feat = feat
        ctx.time = time
        ctx.M_score_func = M_score_func

        # Return ratio_rating_BD as a normal differentiable tensor
        # We'll let PyTorch see it as requires_grad, so chain rule works.
        ratio_rating_BD.requires_grad_(True)
        return ratio_rating_BD

    @staticmethod
    def backward(ctx, grad_output):
        """
        backward: 
         - grad_output is d(Loss)/d(ratio_rating_BD), shape [B, D]
         - We want d(Loss)/d(params_single_tensor).
         - We'll do the forward-mode jac * ratio logic from your snippet.
         - Then we multiply by grad_output for chain rule.
         - We return that as the gradient w.r.t. 'params_single_tensor'.
        """

        # 1) Recover saved items
        params_single_tensor, y_sample_BMD, ratio_rating_BMD = ctx.saved_tensors
        model_build_from_params = ctx.model_build_from_params
        feat = ctx.feat
        time = ctx.time
        M_score_func = ctx.M_score_func

        # 2) We re-run the log_prob under the same distribution, but *as a function of param_tensor*
        def get_log_probs_baked(param_tensor):
            """
            Return log_prob of y_sample_BMD, shape [B, M, D],
            so the result is also [B, M, D].
            """
            distribution = model_build_from_params(param_tensor, feat, time)
            # reorder for log_prob => shape: [M, B, D]
            #print(f'BMD: {y_sample_BMD.size()}')
            y_sample_MBD = y_sample_BMD.permute(1,0,2)  
            log_probs_MBD = distribution.log_prob(y_sample_MBD)
            # permute back => shape: [B, M, D]
            return log_probs_MBD.permute(1,0,2)

        # 3) Compute Jacobian wrt the parameters using forward-mode
        #    shape of jac => [B, M, D, param_dim] if get_log_probs_baked
        #    returns shape [B, M, D].
        jac_BMDP = torch.autograd.functional.jacobian(
            get_log_probs_baked,
            (params_single_tensor,),
            strategy='forward-mode',
            vectorize=True
        )
        # jac_BMDP is a tuple containing the actual tensor if we pass a 1-tuple
        # Typically you'd do: jac_BMDP = jac_BMDP[0] if that is how your code unpacks.

        if isinstance(jac_BMDP, (tuple, list)):
            jac_BMDP = jac_BMDP[0]  # unwrap from tuple

        # 4) Multiply by ratio term => "score function"
        #    ratio_rating_BMD shape [B, M, D], so we broadcast it to [B, M, D, 1]
        sf_BMDP = jac_BMDP * ratio_rating_BMD.unsqueeze(-1)

        # 5) Average over M dimension if that is your desired approach
        #    => shape [B, D, param_dim]
        sf_BDP = sf_BMDP.mean(dim=1)

        # 6) The upstream grad is shape [B, D] => we broadcast to [B, D, 1]
        grad_output_BD1 = grad_output.unsqueeze(-1)

        # 7) chain rule: d(L)/d(param) = sum_{b,d} [ sf_BDP * grad_output_BD1 ]
        #    => shape [param_dim]
        #    You can reduce over B & D by summation
        grad_params = torch.sum(sf_BDP * grad_output_BD1, dim=[0,1])

        # 8) Return gradient w.r.t. each input of `forward`
        # forward signature was (params_single_tensor, feat, time, M_score_func, build_func)
        # so we have to return a gradient for each. We only want to produce non-None
        # for the param tensor. The others we treat as constants => return None.
        d_params_single_tensor = grad_params
        return d_params_single_tensor, None, None, None, None

######################################################
# 2) Example "Model" with single-tensor parameters
######################################################

import torch.nn as nn
from torch.distributions import Categorical, Poisson, MixtureSameFamily, NegativeBinomial


class NegativeBinomialRegressionModel(nn.Module):
    def __init__(self, num_locations, num_fixed_effects, low=0, high=float('inf'), device='cpu'):
        super(NegativeBinomialRegressionModel, self).__init__()
        self.num_locations = num_locations
        self.num_fixed_effects = num_fixed_effects
        self.low = low
        self.high = high
        
        # Fixed effects
        self.beta_0 = nn.Parameter(torch.randn(1))
        self.beta = nn.Parameter(torch.randn(num_fixed_effects))

        # Random effects - why is this not param?
        self.b_0 = nn.Parameter(torch.zeros(num_locations).to(device))
        self.b_1 = nn.Parameter(torch.zeros(num_locations).to(device))

        # Covariance matrix parameters
        self.log_sigma_0 = nn.Parameter(torch.randn(1))
        self.log_sigma_1 = nn.Parameter(torch.randn(1))
        self.rho = nn.Parameter(torch.randn(1))

        # probability param
        self.siginv_theta = nn.Parameter(torch.randn(1))

    def forward(self, X):
        """
        X: Tensor of shape (batch_size, num_locations * (num_fixed_effects + 1))
        The last feature per location is assumed to be 'time'.
        """
        batch_size = X.shape[0]
        expected_features = self.num_fixed_effects + 1  # +1 for time
        total_expected_features = self.num_locations * expected_features

        assert X.shape[1] == total_expected_features, (
            f"Expected input with {total_expected_features} features "
            f"(num_locations={self.num_locations} × {expected_features}), got {X.shape[1]}"
        )

        # Reshape X into (batch_size, num_locations, num_fixed_effects + 1)
        X_reshaped = X.view(batch_size, self.num_locations, expected_features)

        # Split into features and time
        time = X_reshaped[:, :, -1]  # shape: (batch_size, num_locations)
        X_features = X_reshaped[:, :, :-1]  # shape: (batch_size, num_locations, num_fixed_effects)

        assert X_features.shape[2] == self.num_fixed_effects, (
            f"Expected {self.num_fixed_effects} fixed effects, got {X_features.shape[2]}"
        )

        # Calculate fixed effects contribution
        fixed_effects = self.beta_0 + torch.einsum('bli,i->bl', X_features, self.beta)

        # Expand random effects
        random_intercepts = self.b_0.expand(batch_size, -1)
        random_slopes = self.b_1.expand(batch_size, -1)

        # Combine all into log_mu
        log_mu = fixed_effects + random_intercepts + random_slopes * time

        # Use softplus to ensure positivity
        mu = nn.functional.softplus(log_mu)

        # Sigmoid for theta
        theta = torch.sigmoid(self.siginv_theta)

        # Calculate logits
        logits = torch.log(mu) - torch.log(theta)

        # Create NegativeBinomial with logits
        return NegativeBinomial(total_count=mu, probs=theta)
    
    def get_covariance_matrix(self):
        sigma_0 = torch.exp(self.log_sigma_0)
        sigma_1 = torch.exp(self.log_sigma_1)
        rho = torch.tanh(self.rho)  # Ensure -1 < rho < 1
        
        # Construct the covariance matrix using operations that maintain the computational graph
        cov_00 = sigma_0 * sigma_0
        cov_01 = rho * sigma_0 * sigma_1
        cov_11 = sigma_1 * sigma_1
        
        cov_matrix = torch.stack([
            torch.stack([cov_00, cov_01]),
            torch.stack([cov_01, cov_11])
        ])

        cov_matrix = torch.squeeze(cov_matrix)
        
        return cov_matrix

    def log_likelihood(self, y, X, time):
        dist = self.forward(X, time)
        log_prob = dist.log_prob(y)
        
        # Add penalty for random effects
        cov_matrix = self.get_covariance_matrix()
        random_effects = torch.stack([self.b_0, self.b_1], dim=1)
        penalty = -0.5 * torch.mean(
            torch.matmul(random_effects, torch.inverse(cov_matrix)) * random_effects
        )
        
        return torch.mean(log_prob) + penalty

    def sample(self, X, time, num_samples=1):
        dist = self.forward(X, time)
        samples = dist.sample((num_samples,))
        return torch.clamp(samples, self.low, self.high)

    def params_to_single_tensor(self):
        return torch.cat([param.view(-1) for param in self.parameters()])

    def single_tensor_to_params(self, single_tensor):
        params = []
        idx = 0
        for param in self.parameters():
            num_elements = param.numel()
            param_data = single_tensor[idx:idx+num_elements].view(param.shape)
            params.append(param_data)
            idx += num_elements
        return params

    def update_params(self, single_tensor):
        params = self.single_tensor_to_params(single_tensor)
        for param, new_data in zip(self.parameters(), params):
            param.data = new_data

    def build_from_single_tensor(self, single_tensor, X, time):
        beta_0, beta, b_0, b_1, log_sigma_0, log_sigma_1, rho, siginv_theta = self.single_tensor_to_params(single_tensor)
        fixed_effects = beta_0 + torch.einsum('tli,i->tl', X, beta)
        random_intercepts = b_0.expand(X.shape[0], -1)
        random_slopes = b_1.expand(X.shape[0], -1)
        
        log_mu = fixed_effects + random_intercepts + random_slopes * time

        # Use softplus to ensure mu is positive and grows more slowly
        mu = nn.functional.softplus(log_mu)

        # Calculate theta probability
        theta = torch.nn.functional.sigmoid(siginv_theta)

        
        return NegativeBinomial(total_count=mu, probs=theta)

    def ratio_rating(self, X, num_samples=2):
        """
        This is the user-facing call that yields ratio_rating_BD 
        as a normal PyTorch tensor. Internally it calls the 
        custom autograd function.
        """
        batch_size = X.shape[0]
        expected_features = self.num_fixed_effects + 1  # +1 for time
        total_expected_features = self.num_locations * expected_features

        assert X.shape[1] == total_expected_features, (
            f"Expected input with {total_expected_features} features "
            f"(num_locations={self.num_locations} × {expected_features}), got {X.shape[1]}"
        )

        # Reshape X into (batch_size, num_locations, num_fixed_effects + 1)
        X_reshaped = X.view(batch_size, self.num_locations, expected_features)

        # Split into features and time
        time = X_reshaped[:, :, -1]  # shape: (batch_size, num_locations)
        X_features = X_reshaped[:, :, :-1]  # shape: (batch_size, num_locations, num_fixed_effects)

        params_single_tensor = self.params_to_single_tensor()
        ratio_rating_BD = RatioRatingFunction.apply(
            params_single_tensor,
            X_features,
            time,
            num_samples,
            self.build_from_single_tensor
        )
        return ratio_rating_BD

######################################################
# 3) Example usage with a custom BPR-ish loss
######################################################

if False:
    # Suppose param_dim=5, just a toy
    model = MyModel(param_dim=5)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

    for epoch in range(3):
        optimizer.zero_grad()

        # Fake "feat, time" for the example
        feat = torch.randn(4,10)  # shape whatever
        time = torch.tensor([0])  # shape ???

        # ratio_rating_BD => shape [B, D], let's say [1,5] if we want
        ratio_rating_BD = model.ratio_rating(feat, time, M_score_func=10)

        # Now do your normal PyTorch code for BPR or whatever:
        # e.g. let's make a dummy loss that sums ratio_rating_BD**2
        # Obviously you'd do your real BPR or other partial derivatives, etc.
        loss = (ratio_rating_BD**2).mean()

        # Because ratio_rating_BD is a normal differentiable tensor,
        # we can do standard autograd
        loss.backward()    # This triggers RatioRatingFunction's backward
        optimizer.step()

        print(f"Epoch {epoch}: loss={loss.item():.4f}")


In [2]:
# train model
def trainModel(prediction_model, loss_func, method_name, num_epochs=40, lr=1e-1):
    # set adam optimizer
    optimizer = torch.optim.Adam(prediction_model.parameters(), lr=lr)
    # train mode
    prediction_model.train()
    # log
    #loss_log, regret_log = [], [pyepo.metric.regret(prediction_model, optmodel, loader_test)]
    loss_log = []
    for epoch in range(num_epochs):
        # load data
        for i, data in enumerate(loader_train):
            print(f'i: {i}')
            x, c, w, z = data
            # cuda
            if torch.cuda.is_available():
                x, c, w, z = x.cuda(), c.cuda(), w.cuda(), z.cuda()

            
            # forward pass
            cp = prediction_model.ratio_rating(x, num_samples=100)
            print(f' Got the predicted cost, shape: {cp.size()}')
            if method_name == "spo+":
                loss = loss_func(cp, c, w, z)
            elif method_name in ["ptb", "pfy", "imle", "nce"]:
                loss = loss_func(cp, w)
            elif method_name in ["dbb", "nid"]:
                loss = loss_func(cp, c, z)
            elif method_name in ["2s", "pg", "ltr"]:
                loss = loss_func(cp, c)
            # record loss
            loss_log.append(loss.item())
            print(f'Got the loss: {loss.item()}')
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # record regrer
        if epoch % 2 == 0:
            #regret = pyepo.metric.regret(prediction_model, optmodel, loader_test)
            print("Epoch {:2},  Loss: {:9.4f},  %".format(epoch, loss.item(), ))#regret*100))
            #regret_log.append(regret)
    # plot
    visLearningCurve(loss_log, [])

In [3]:
model = NegativeBinomialRegressionModel(num_locations=m, num_fixed_effects=p-1, device='cuda').to('cuda')

NameError: name 'm' is not defined

In [20]:
# init SPO+ loss
spop = pyepo.func.SPOPlus(optmodel, processes=8)
loss_func = spop 
method_name = "spo+"

Num of cores: 8


In [21]:
# init SPO+ loss
pg = pyepo.func.perturbationGradient(optmodel, sigma=1.0, two_sides=False, processes=1)
loss_func = pg
method_name = "pg"

Num of cores: 1


In [ ]:
trainModel(model, loss_func, method_name, num_epochs=40, lr=1e-5)

i: 0
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.19073882699012756
BMD: torch.Size([1, 100, 275])
i: 1
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.19381564855575562
BMD: torch.Size([1, 100, 275])
i: 2
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.20379573106765747
BMD: torch.Size([1, 100, 275])
i: 3
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.22908411920070648
BMD: torch.Size([1, 100, 275])
i: 4
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.20982035994529724
BMD: torch.Size([1, 100, 275])
i: 5
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.20003928244113922
BMD: torch.Size([1, 100, 275])
i: 6
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.19224730134010315
BMD: torch.Size([1, 100, 275])
i: 7
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.20493820309638977
BMD: torch.Size([1, 100, 275])
i: 8
 Go

Got the loss: -0.21095803380012512
BMD: torch.Size([1, 100, 275])
i: 3
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.21539483964443207
BMD: torch.Size([1, 100, 275])
i: 4
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.2198355793952942
BMD: torch.Size([1, 100, 275])
i: 5
 Got the predicted cost, shape: torch.Size([1, 275])

Interrupt request received
Got the loss: -0.2080720216035843
BMD: torch.Size([1, 100, 275])
i: 6
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.19602367281913757
BMD: torch.Size([1, 100, 275])
i: 7
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.20431914925575256
BMD: torch.Size([1, 100, 275])
i: 8
 Got the predicted cost, shape: torch.Size([1, 275])
Got the loss: -0.21096393465995789
BMD: torch.Size([1, 100, 275])
i: 9
 Got the predicted cost, shape: torch.Size([1, 275])


In [59]:
x_train.device

'cpu'

In [34]:
data

NameError: name 'data' is not defined

In [40]:
weights.shape

(1, 1620)

In [39]:
c

array([[ 2.,  2.,  3., ..., 10.,  3.,  2.],
       [ 3.,  3.,  4., ...,  3.,  2.,  5.],
       [ 4.,  3.,  7., ...,  6.,  2., 10.],
       ...,
       [ 1., 12.,  4., ..., 25., 15.,  1.],
       [ 1.,  5.,  3., ...,  4.,  5.,  1.],
       [ 4.,  5.,  3., ...,  2.,  3.,  6.]], shape=(1000, 1620))